# Module 03: Interactive Data Structures & Memory Internals

### What You Will Discover
By running this notebook, you will explore how Python stores collections under the hood: list over-allocation growth patterns, dictionary compact hash table indices, and memory costs.

**Key Question Answered:** *Why does appending 1 element to a list sometimes consume 0 additional bytes of OS memory, while other times it allocates dozens of bytes at once?*


In [ ]:
# Step 1: Base list creation and sys.getsizeof measurement
import sys

items = []
print(f'Empty list base size: {sys.getsizeof(items)} bytes')


In [ ]:
# Step 2: Append elements and observe how size jumps in discrete blocks
sizes = []
for i in range(10):
    items.append(i)
    sizes.append((len(items), sys.getsizeof(items)))

for length, size in sizes[:5]:
    print(f'Length: {length:2d} -> Bytes: {size}')


In [ ]:
# Step 3: Inspect the remaining growth trajectory
for length, size in sizes[5:]:
    print(f'Length: {length:2d} -> Bytes: {size}')


### 🔮 Prediction Prompt
**Before running the next cell:** If you create a list of 100 identical integers using `[0] * 100`, vs repeatedly appending 100 times with `.append(0)`, will both lists occupy the exact same number of bytes in memory? Write down your guess!


In [ ]:
# Surprising Result: Pre-sized multiplication vs dynamic append growth
pre_allocated = [0] * 100
dynamic_append = []
for _ in range(100):
    dynamic_append.append(0)

print(f'Pre-allocated [0]*100 size : {sys.getsizeof(pre_allocated)} bytes')
print(f'Dynamic append 100x size   : {sys.getsizeof(dynamic_append)} bytes')
print('Explanation: list.append() over-allocates extra capacity to achieve amortized O(1) appends!')


### Dictionary Internals: Compact Hash Tables
Since Python 3.6 (PEP 468), dictionaries are ordered and compact. Let us inspect key lookup speed and hashing.


In [ ]:
# Inspecting dictionary memory vs set memory
sample_dict = {f'key_{i}': i for i in range(100)}
sample_set = {f'key_{i}' for i in range(100)}

print(f'Dict 100 keys: {sys.getsizeof(sample_dict)} bytes')
print(f'Set 100 items:  {sys.getsizeof(sample_set)} bytes')


### High-Performance Collections: `collections.deque` vs `list.pop(0)`
Popping from the front of a standard list is $O(N)$ because every remaining pointer must shift down. A `deque` is a doubly-linked block list with $O(1)$ front pops.


In [ ]:
import collections
import time

dq = collections.deque(range(50_000))
start = time.perf_counter()
for _ in range(10_000):
    dq.popleft()
elapsed_dq = time.perf_counter() - start
print(f'Deque 10,000 popleft(): {elapsed_dq * 1000:.2f} ms (O(1) per op)')


### 🛠️ Interactive Challenge: Fix the Inefficient Queue
The following cell attempts to implement a FIFO queue using a standard Python `list`. When processing high volumes, it suffers severe $O(N^2)$ slowdown. Fix the implementation to use `collections.deque` so it executes in sub-millisecond time.


In [ ]:
# TODO: FIX ME - Replace naive list FIFO with collections.deque to achieve O(1) pops
# Currently this implementation uses list.pop(0) which is O(N)

class NaiveFIFOQueue:
    def __init__(self):
        # FIX: self._storage = collections.deque()
        self._storage = []

    def push(self, item):
        self._storage.append(item)

    def pop(self):
        # FIX: return self._storage.popleft()
        return self._storage.pop(0)  # Intentionally inefficient O(N) operation

q = NaiveFIFOQueue()
for i in range(100):
    q.push(i)
print(f'Popped first item: {q.pop()}')


### 🏁 Summary & Next Steps
- Lists over-allocate in discrete buckets: $0, 4, 8, 16, 24, 32, 40, 52, 64 \dots$
- Dictionaries use compact indices pointing into sparse hash arrays.
- Next: run `python 01_lists_and_tuples_internals_demo.py` and `python 02_dicts_and_sets_hash_demo.py`.
- Work through [PROJECT_GUIDE.md](PROJECT_GUIDE.md) to implement the high-concurrency LRU cache.
